# Supply Chain Optimisation using Mixed-Integer Linear Programming (MILP)
## Introduction
Welcome to this Jupyter Notebook! In this notebook, we will explore the application of Mixed-Integer Linear Programming (MILP) to solve a supply chain optimisation problem. Supply chain management involves the coordination and integration of various activities such as procurement, production, inventory management, and distribution to maximize efficiency and minimize costs.

MILP is a powerful optimisation technique that allows us to model complex decision-making processes involving both continuous and discrete variables. By formulating the supply chain problem as a MILP model, we can optimize various aspects such as production scheduling, inventory levels, and transportation routes to achieve the best possible outcomes.

## Throughout this notebook, we will:
1. Define the supply chain problem and its objectives.
2. Formulate the MILP model, including decision variables, constraints, and the objective function.
3. Implement the MILP model using a suitable optimisation solver.
4. Analyze the results and interpret the optimal solution.

By the end of this notebook, you will have a clear understanding of how MILP can be applied to solve real-world supply chain problems, and you will gain hands-on experience in implementing and solving such models.

# Problem Description
In this supply chain optimisation problem, the primary objective is to allocate stock to different sites in order to build systems that generate revenue. Each site has specific configurations for the systems they can build, and each system brings different revenues depending on the site. The goal is to maximize the total revenue while considering various costs and constraints.

## Key Components of the Problem
- Sites: There are multiple sites where systems can be built. Each site has its own specificities and demands.
- Systems: Each site can build different systems, and each system configuration is unique to the site. Once completed, the systems are shipped to the customer and generate monthly revenues. The revenue generated by each system varies.
- Stock Allocation: The stock needs to be allocated to the sites to enable the building of systems. The allocation should meet the demand for each system at each site. The revenues generated by the systems that exceed initial demand are lower. This will take the form of a penalty.
- Delivery Cost: There is a fee associated with delivering the stock to the sites, referred to as "delivery_per_unit." This cost must be factored into the optimisation process.

## Objective
The primary objective is to maximize the total revenue generated by building systems across all sites. This involves:
- Allocating the available stock to the sites in such a way that the demand for each system is met.
- Minimizing the delivery costs associated with transporting the stock to the sites.
- Ensuring that the systems built at each site generate the maximum possible revenue.

## Constraints
- Stock Availability: The total stock allocated to all sites must not exceed the available stock.
- Demand Satisfaction: The allocation should meet the specific demand for each system at each site.
- Delivery Cost: The cost of delivering the stock to the sites must be considered in the optimisation process.

## Decision Variables
- Allocation Variables: Decision variables that represent the amount of stock allocated to each site.
- Build Variables: Decision variables that represent the number of each system built at each site.
- A slack variable will be used to modelise the penalty when there are more systems completed than the demand.

# Formulation of MILP model

In [1]:
import pandas as pd

import pulp
pulp.listSolvers(onlyAvailable=True)  # We use the standard solver provided by PuLP

['PULP_CBC_CMD']

The 4 sites of the model, with inbounds costs per component, and the loss of revenues for each system above initial demand.

In [2]:
sites = (
    pd.DataFrame(
        [
            ['BCN', 1.50, 0.11],
            ['MIL', 1.50, 0.18],
            ['ZCH', 1.00, 0.17],
            ['WRO', 1.20, 0.13]
        ],
        columns=['site', 'delivery_per_unit', 'exceed_demand_penalty']
    )
    .set_index('site')
)
sites

,delivery_per_unit,exceed_demand_penalty
site,,
BCN,1.5,0.11
MIL,1.5,0.18
ZCH,1.0,0.17
WRO,1.2,0.13


The monthly revenues generated by each system depends on the site it's shipped from. Not all systems are shipped from all sites.  
Each site has information from its customer base and know how many systems it can ship at full revenues. Systems exceeding this amount will generate less revenues, this is modelised as a penalty in the model - see above.

In [3]:
sites_systems = (
    pd.DataFrame(
        [
            ['BCN', 'sysA', 2000, 50],
            ['BCN', 'sysB', 1300, 10],
            ['BCN', 'sysC', 3700, 40],
            ['MIL', 'sysA', 2200, 35],
            ['MIL', 'sysB', 2100, 30],
            ['MIL', 'sysC', 4000, 50],
            ['ZCH', 'sysA', 3400, 15],
            ['ZCH', 'sysC', 4700, 10],
            ['WRO', 'sysA', 1800, 25],
            ['WRO', 'sysB', 3600, 5],
            ['WRO', 'sysC', 2300, 20],
        ],
        columns=['site', 'system', 'monthly_rev', 'demand']
    )
    .set_index(['site', 'system'])
)
sites_systems

monthly_rev  demand
site system                     
BCN  sysA           2000      50
     sysB           1300      10
     sysC           3700      40
MIL  sysA           2200      35
     sysB           2100      30
     sysC           4000      50
ZCH  sysA           3400      15
     sysC           4700      10
WRO  sysA           1800      25
     sysB           3600       5
     sysC           2300      20

We define which components are required to complete systems. This is specific to each site.

In [4]:
list_matreqs = lambda site, sys, matreqs: [[site, sys, mat, qty] for mat, qty in matreqs.items()]

sites_systems_matreq = (
    pd.DataFrame(
        [
            *list_matreqs('BCN', 'sysA', {'m0': 1, 'm1': 1, 'm2':1}),
            *list_matreqs('MIL', 'sysA', {'m0': 1, 'm1': 1, 'm2':1}),
            *list_matreqs('ZCH', 'sysA', {'m0': 2, 'm1': 1, 'm2':1}),
            *list_matreqs('WRO', 'sysA', {'m0': 1, 'm1': 1, 'm2':1}),
            *list_matreqs('BCN', 'sysB', {'m3': 2, 'm4': 1, 'm5':1}),
            *list_matreqs('MIL', 'sysB', {'m3': 1, 'm4': 1, 'm5':2}),
            *list_matreqs('WRO', 'sysB', {'m2': 3, 'm3': 2, 'm5':1}),
            *list_matreqs('BCN', 'sysC', {'m1': 1, 'm5': 1, 'm6':1}),
            *list_matreqs('MIL', 'sysC', {'m0': 1, 'm5': 1, 'm6':1}),
            *list_matreqs('ZCH', 'sysC', {'m0': 1, 'm1': 1, 'm5': 1, 'm6': 1}),
            *list_matreqs('WRO', 'sysC', {'m1': 1, 'm5': 1, 'm6': 2}),
        ],
        columns=['site', 'system', 'matreq', 'qty']
    )
    .set_index(['site', 'system', 'matreq']).sort_index()
)
sites_systems_matreq.pivot_table(index=['system', 'site'], columns='matreq').fillna('')

qty                              
matreq        m0   m1   m2   m3   m4   m5   m6
system site                                   
sysA   BCN   1.0  1.0  1.0                    
       MIL   1.0  1.0  1.0                    
       WRO   1.0  1.0  1.0                    
       ZCH   2.0  1.0  1.0                    
sysB   BCN                  2.0  1.0  1.0     
       MIL                  1.0  1.0  2.0     
       WRO             3.0  2.0       1.0     
sysC   BCN        1.0                 1.0  1.0
       MIL   1.0                      1.0  1.0
       WRO        1.0                 1.0  2.0
       ZCH   1.0  1.0                 1.0  1.0

We define how many units of material are available.

In [5]:
avail_mat = (
    pd.DataFrame(
        [
            [f'm{i}', qty] for i, qty in enumerate([160, 160, 140, 100, 70, 150, 90])
        ],
        columns=['material', 'qty']
    )
    .set_index(['material'])
)
avail_mat.T

material,m0,m1,m2,m3,m4,m5,m6
qty,160,160,140,100,70,150,90


In [6]:
all_materials = sites_systems_matreq.index.droplevel([0,1]).drop_duplicates().to_list()
all_sites_systems_materials = sites_systems_matreq.index.to_list()
all_sites_systems = sites_systems_matreq.index.droplevel(2).drop_duplicates().to_list()

## Variables

In [7]:
# Variable: number of materials allocation by site & system
v_mat_allocated = pulp.LpVariable.dicts(
    name='Material allocated by site & system',
    indices=all_sites_systems_materials,
    cat='Integer',
    lowBound=0
)

In [8]:
# Variable: number of systems that are complete
v_sys_complete = pulp.LpVariable.dicts(
    name='Complete system by site',
    indices=all_sites_systems,
    cat='Integer',
    lowBound=0
)

We add a slack variable to establish a soft constraint on the number of systems. Later on, a penalty is applied in the objective function for each system exceeding demand quantity.

In [9]:
# Slack variable: Number of systems above demand
v_sys_exceed_demand = pulp.LpVariable.dicts(
    name='Complete systems exceeding demand',
    indices=all_sites_systems,
    cat='Integer',
    lowBound=0
)

## Objective function
It contains 3 elements:
1. The monthly revenues generated by all completed systems
2. The inbound cost
3. The penalty applied for each system above initial demand

In [10]:
# Objective function
prob = pulp.LpProblem('AchieveBestRevenuesMinusCosts', pulp.LpMaximize)
prob += (
    pulp.lpSum([
        v_sys_complete[site, system] * sites_systems.loc[(site, system), 'monthly_rev']
        for (site, system) in all_sites_systems
    ])
    - pulp.lpSum([
        v_mat_allocated[site, system, material] * sites.loc[site, 'delivery_per_unit']
        for (site, system, material) in all_sites_systems_materials
    ])
    - pulp.lpSum([
        v_sys_exceed_demand[site, system] * sites.loc[site, 'exceed_demand_penalty'] * sites_systems.loc[(site, system), 'monthly_rev']
        for (site, system) in all_sites_systems
    ]),
    'Total revenue after costs'
)

## Constraints

In [11]:
# Constraint: We don't allocate more material quantity than available
for material in all_materials:
    prob += (
        pulp.lpSum([
            v_mat_allocated[site, system, material]
            for (site, system) in all_sites_systems
            if material in sites_systems_matreq.loc[(site, system)].index.to_list()
        ])
        <= avail_mat.loc[material],
        f"Don't allocate more {material} than available"
    )

In [12]:
# Constraint: Number of systems completed by site
for (site, system) in all_sites_systems:
    for material, req_qty in sites_systems_matreq.loc[(site, system)].to_records():
        prob += (
            req_qty * v_sys_complete[site, system] <= v_mat_allocated[site, system, material],
            f'No. systems {site}/{system} limited by {material}'
        )

In [13]:
# Constraint: Lower bound for slack variable No. systems exceeding demand
for (site, system) in all_sites_systems:
    prob += (
        (v_sys_complete[site, system] - v_sys_exceed_demand[site, system]) <= sites_systems.loc[(site, system), 'demand'],
        f'No. systems {site}/{system} limited by demand'
    )

## Solving & Result output
Our problem is mixed integer, linear, non-quadratic and therefore is suitable to PuLP standard solver PULP_CBC_CMD.  
Before solving the model, it's important to note that this solver is freely available and suitable for many optimization problems. However, for more complex or large-scale problems, using a commercial solver like Gurobi could provide significant advantages, including faster solving times, better handling of large datasets, and more advanced algorithms. Nevertheless, our current problem can be effectively resolved using PULP_CBC_CMD.

In [14]:
# prob.writeLP("OptimiseRevs.lp")
prob.solve()
print(f'Status: {pulp.LpStatus[prob.status]}, achieved {pulp.value(prob.objective)}')

Status: Optimal, achieved 677290.9


In [15]:
# Collecting optimal results in dataframes
summary_material_allocations = (
    pd.Series(
        data = [int(v_mat_allocated[*ssm].varValue) for ssm in all_sites_systems_materials],
        index = pd.MultiIndex.from_tuples(all_sites_systems_materials, names=('site', 'system', 'material')),
        name = 'Material allocated'
    )
    .to_frame()
)
summary_systems_created = (
    pd.Series(
        data = [int(v_sys_complete[*ss].varValue) for ss in all_sites_systems],
        index = pd.MultiIndex.from_tuples(all_sites_systems, names=('site', 'system')),
        name = 'Systems complete'
    )
    .to_frame()
)

We display the results in a human-readable format, starting with materials allocations by site.

In [16]:
def int_format(an_int):
    if an_int == 0:
        return ''
    return f'{int(an_int):d}' 

display(
    summary_material_allocations
    .pivot_table(index=['site', 'system'], columns='material', aggfunc='sum', fill_value=0)
    .droplevel(0, axis=1)
    .pipe(
        lambda df: pd.concat(
            [
                df,
                df.sum(axis=0).rename(('Total', 'allocated')).to_frame().T,
                avail_mat['qty'].rename(('Total', 'available')).to_frame().T
            ],
            axis=0
        )
    )
    .pipe(
        lambda df: pd.concat(
            [
                df,
                (df.loc[('Total', 'available')] - df.loc[('Total', 'allocated')]).rename(('Total', 'unused')).to_frame().T
            ],
            axis=0
        )
    )
    .map(int_format)
)

material          m0   m1   m2   m3  m4   m5  m6
BCN   sysA        48   48   48                  
      sysB                       66  33   33    
      sysC             49                 49  49
MIL   sysA        35   35   35                  
      sysB                        7   7   14    
      sysC        31                      31  31
WRO   sysA                                      
      sysB                  39   26       13    
      sysC                                      
ZCH   sysA        36   18   18                  
      sysC        10   10                 10  10
Total allocated  160  160  140   99  40  150  90
      available  160  160  140  100  70  150  90
      unused                      1  30

We summarize  

In [17]:
display(
    summary_systems_created
    .join(sites_systems, how='left')
    .join(sites, how='left')
    .join(summary_material_allocations.pivot_table(index=['site', 'system'], aggfunc='sum'))
    .assign(
        num_systems_vs_target = lambda df: df['Systems complete'] - df['demand'],
        total_revenues = lambda df: df['Systems complete']*df['monthly_rev'],
        penalty_demand_exceeded = lambda df: df['num_systems_vs_target'].clip(0)*df['exceed_demand_penalty']*df['monthly_rev'],
        total_costs = lambda df: df['delivery_per_unit'] * df['Material allocated'],
        total_profit = lambda df: df['total_revenues'] - df['total_costs'] - df['penalty_demand_exceeded']
    )
    .pipe(
        lambda df: pd.concat([df, df.sum(axis=0).rename(('Total', 'allocated')).to_frame().T], axis=0)
    )
    .T
)

BCN                            MIL            \
                             sysA      sysB       sysC      sysA      sysB   
Systems complete            48.00     33.00      49.00     35.00      7.00   
monthly_rev               2000.00   1300.00    3700.00   2200.00   2100.00   
demand                      50.00     10.00      40.00     35.00     30.00   
delivery_per_unit            1.50      1.50       1.50      1.50      1.50   
exceed_demand_penalty        0.11      0.11       0.11      0.18      0.18   
Material allocated         144.00    132.00     147.00    105.00     28.00   
num_systems_vs_target       -2.00     23.00       9.00      0.00    -23.00   
total_revenues           96000.00  42900.00  181300.00  77000.00  14700.00   
penalty_demand_exceeded      0.00   3289.00    3663.00      0.00      0.00   
total_costs                216.00    198.00     220.50    157.50     42.00   
total_profit             95784.00  39413.00  177416.50  76842.50  14658.00   

                                        WRO                          ZCH  \
                              sysC     sysA      sysB     sysC      sysA   
Systems complete             31.00     0.00     13.00     0.00     18.00   
monthly_rev                4000.00  1800.00   3600.00  2300.00   3400.00   
demand                       50.00    25.00      5.00    20.00     15.00   
delivery_per_unit             1.50     1.20      1.20     1.20      1.00   
exceed_demand_penalty         0.18     0.13      0.13     0.13      0.17   
Material allocated           93.00     0.00     78.00     0.00     72.00   
num_systems_vs_target       -19.00   -25.00      8.00   -20.00      3.00   
total_revenues           124000.00     0.00  46800.00     0.00  61200.00   
penalty_demand_exceeded       0.00     0.00   3744.00     0.00   1734.00   
total_costs                 139.50     0.00     93.60     0.00     72.00   
total_profit             123860.50     0.00  42962.40     0.00  59394.00   

                                      Total  
                             sysC allocated  
Systems complete            10.00     244.0  
monthly_rev               4700.00   31100.0  
demand                      10.00     290.0  
delivery_per_unit            1.00      14.6  
exceed_demand_penalty        0.17       1.6  
Material allocated          40.00     839.0  
num_systems_vs_target        0.00     -46.0  
total_revenues           47000.00  690900.0  
penalty_demand_exceeded      0.00   12430.0  
total_costs                 40.00    1179.1  
total_profit             46960.00  677290.9

# Analysing the results & Interpreting the optimal solution

The Mixed-Integer Linear Programming (MILP) model successfully generated a feasible solution, providing the optimal allocation of stock to different sites for building systems. However, it is important to note several key observations:

## Demand Satisfaction
The model indicates that not all demand could be met due to a lack of materials. This highlights the constraint imposed by limited resources, which prevents full demand satisfaction across all sites.

## Excess Materials
Conversely, there are instances where certain materials are in excess (m4). This suggests that there may be opportunities for better resource inbound management to maximize overall efficiency.

## Excess System Completion
There are a few occurrences where the number of completed systems exceeds the demand at specific sites. These excess systems are subject to a fee, representing the reduced revenue generated from these additional units. This scenario is made possible through the use of slack variables in the model, which allow for flexibility in meeting demand while considering the trade-offs between revenue and excess production costs.  
The use of slack variables enables the model to find a balance between maximizing revenue and managing excess production. This approach ensures that the solution is both feasible and optimal, even when demand cannot be fully met.

# Concluding

## About solvers
In this exercise, we utilized the PuLP library with its standard solver, PULP_CBC_CMD, to solve the MILP problem. While PULP_CBC_CMD is a robust and freely available solver, there are more advanced solvers like Gurobi and GAMS that could offer several advantages:
- Performance: Gurobi and GAMS are known for their high performance and efficiency, often solving complex problems faster than open-source solvers. This can be particularly beneficial for large-scale supply chain optimisation problems.
- Advanced Features: These commercial solvers offer advanced features such as better handling of large datasets, more sophisticated algorithms for finding optimal solutions, and enhanced support for parallel computing.
- Scalability: For industrial-scale problems, the scalability and reliability of commercial solvers can be crucial, ensuring that the model can handle increasing complexity and data volume.

## Going forward
Overall, the model provides valuable insights into the supply chain's constraints and opportunities. Future iterations of the model could focus on several improvements to enhance its practicality and effectiveness. These include:
- Considering Existing Stock: Incorporating the existing stock levels at each site into the model to better utilize available resources and reduce the need for additional deliveries.
- Weekly Runs: Allowing the model to be run on a weekly basis to adapt to changing demands and material availability, ensuring more dynamic and responsive supply chain management.
- Demand Forecasting: Integrating demand forecasting techniques to anticipate future needs and adjust stock allocation accordingly.
- Cost Optimisation: Exploring additional cost-saving measures, such as negotiating better delivery rates or optimizing production schedules.
By implementing these enhancements, the model can provide even more robust and actionable insights, leading to improved supply chain efficiency and increased revenue generation.